# FairML Toolkit — Part 5: Pipeline Orchestrator Demo

This notebook walks through one complete execution of the **Fairness Pipeline Development Toolkit**.

The pipeline is fully defined by `config.yml` and executed by `run_pipeline.py`.  
No code changes are needed between runs — only the config changes.

---

## What we will do

| Step | Module | What happens |
|---|---|---|
| 1 | MeasurementModule | Audit raw label disparity → **baseline report** |
| 2a | PipelineModule | Apply `DisparateImpactRemover` to repair feature distributions |
| 2b | TrainingModule | Train a `LogisticRegression` under a `DemographicParity` constraint |
| 3 | MeasurementModule | Re-evaluate on test predictions → **report card** |
| — | MLflow | Log all metrics and artifacts for reproducibility |

## 0. Environment check

In [1]:
import importlib, sys

required = ["yaml", "mlflow", "fairlearn", "sklearn", "pandas", "numpy"]
missing  = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    print(f"Missing packages: {missing}")
    print("Run:  pip install -r requirements.txt")
else:
    print("✓ All required packages are installed.")
    print(f"  Python {sys.version.split()[0]}")

✓ All required packages are installed.
  Python 3.11.9


---

## 1. Inspect `config.yml`

The config is the **single source of truth** for the entire pipeline.  
Every transformer, constraint, threshold, and MLflow setting lives here.

In [2]:
import yaml
from pathlib import Path

CONFIG_PATH = "config.yml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

print(Path(CONFIG_PATH).read_text())

# config.yml
# ==========
# FairML Toolkit â€” Part 5: Pipeline Orchestrator Configuration
#
# This file fully defines one reproducible fair ML workflow.
# Edit the values below to switch datasets, transformers, or constraints
# without touching any code.

# ---------------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------------
data:
  path: "data/loan_dataset.csv"
  target_col: "Loan_Approval_Status"
  sensitive_col: "Gender_Male"      # must be numeric (0/1) after preprocessing
  positive_label: 1
  drop_cols: []                     # columns to discard before anything runs

# ---------------------------------------------------------------------------
# Raw preprocessing
# Runs before the fairness transformer so the dataset is in the expected shape.
# ---------------------------------------------------------------------------
preprocessing:
  encode:
    - column: "Gender"              # one-

### Key settings at a glance

In [3]:
import pandas as pd

summary = {
    "Dataset":           cfg["data"]["path"],
    "Target column":     cfg["data"]["target_col"],
    "Sensitive column":  cfg["data"]["sensitive_col"],
    "Transformer":       cfg["transformer"]["class"],
    "Repair level":      cfg["transformer"]["params"].get("repair_level"),
    "Training method":   cfg["training"]["method"],
    "Constraint":        cfg["training"]["constraint"],
    "Primary metric":    cfg["validation"]["primary_metric"],
    "Pass threshold":    cfg["validation"]["threshold"],
    "MLflow experiment": cfg["mlflow"]["experiment_name"],
}

pd.DataFrame.from_dict(summary, orient="index", columns=["Value"])

,Value
Dataset,data/loan_dataset.csv
Target column,Loan_Approval_Status
Sensitive column,Gender_Male
Transformer,DisparateImpactRemover
Repair level,0.8
Training method,ReductionsWrapper
Constraint,DemographicParity
Primary metric,demographic_parity_difference
Pass threshold,0.1
MLflow experiment,fairness_pipeline_run


---

## 2. Load and preview the dataset

In [4]:
from run_pipeline import load_config, load_data

cfg = load_config(CONFIG_PATH)
df  = load_data(cfg)

print(f"\nShape : {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

✓ Data loaded: 52,000 rows × 27 columns.

Shape : (52000, 27)
Columns: ['Applicant_ID', 'Age', 'Marital_Status', 'Dependents', 'Education', 'Employment_Status', 'Occupation_Type', 'Residential_Status', 'City/Town', 'Annual_Income', 'Monthly_Expenses', 'Credit_Score', 'Existing_Loans', 'Total_Existing_Loan_Amount', 'Outstanding_Debt', 'Loan_History', 'Loan_Amount_Requested', 'Loan_Term', 'Loan_Purpose', 'Interest_Rate', 'Loan_Type', 'Co-Applicant', 'Bank_Account_History', 'Transaction_Frequency', 'Default_Risk', 'Loan_Approval_Status', 'Gender_Male']


,Applicant_ID,Age,Marital_Status,Dependents,Education,Employment_Status,Occupation_Type,Residential_Status,City/Town,Annual_Income,...,Loan_Term,Loan_Purpose,Interest_Rate,Loan_Type,Co-Applicant,Bank_Account_History,Transaction_Frequency,Default_Risk,Loan_Approval_Status,Gender_Male
0,1,25,Married,2,Graduate,Employed,Business,Own,Urban,139901,...,209,Home,4.27,Secured,Yes,8,20,0.81,1,0
1,2,36,Married,2,High School,Employed,Business,Own,Suburban,21162,...,33,Home,14.78,Unsecured,Yes,9,9,0.17,0,1
2,3,43,Single,0,Postgraduate,Self-Employed,Freelancer,Own,Urban,27815,...,159,Vehicle,12.33,Secured,Yes,7,27,0.25,0,1
3,4,28,Married,0,High School,Self-Employed,Freelancer,Rent,Suburban,137853,...,39,Personal,8.77,Secured,No,9,16,0.27,1,0
4,5,32,Single,0,Graduate,Employed,Salaried,Rent,Suburban,81753,...,34,Home,9.04,Unsecured,No,1,17,0.32,1,0


In [5]:
sensitive_col = cfg["data"]["sensitive_col"]
target_col    = cfg["data"]["target_col"]

print("Group distribution (sensitive attribute):")
print(df[sensitive_col].value_counts())
print()
print("Positive-label rate by group:")
print(
    df.groupby(sensitive_col)[target_col]
    .mean()
    .rename("approval_rate")
    .round(4)
)

Group distribution (sensitive attribute):
Gender_Male
1    26011
0    25989
Name: count, dtype: int64

Positive-label rate by group:
Gender_Male
0    0.6437
1    0.6397
Name: approval_rate, dtype: float64


---

## 3. Step 1 — Baseline fairness audit

`FairnessAnalyzer` measures label-level disparity in the **raw data** before any intervention.  
This is our starting point — the bias we need to reduce.

In [6]:
from run_pipeline import step1_baseline

baseline_results = step1_baseline(df, cfg)


STEP 1 — BASELINE FAIRNESS AUDIT
FairnessAnalyzer ready — 52,000 rows loaded.


c:\Users\gloccioni\AI-Ethics-Project\run_pipeline.py:134: UserWarning: y_pred not provided – using target column as proxy. Metrics will reflect label disparities, not model bias.
  results = analyzer.calculate_classification_metrics(



  Demographic Parity Difference
  Value       : 0.0040
  95% CI      : (0.0003, 0.0120)
  Effect size : 0.9937785536822049
  Groups (n)  : {1: 26011, 0: 25989}


In [7]:
primary = cfg["validation"]["primary_metric"]
b       = baseline_results[primary]

print("Baseline FairnessResult:")
print(b)

Baseline FairnessResult:
FairnessResult(Demographic Parity Difference)
  value              = 0.0040
  95% CI             = (0.0003, 0.0120)
  effect_size        = 0.9937785536822049
  sample_sizes       = {1: 26011, 0: 25989}



**Interpreting the baseline:**

- **Value** — the raw demographic parity gap between groups. A value of 0 means perfect parity; larger absolute values indicate greater disparity.
- **95% CI** — bootstrapped confidence interval. If it excludes zero, the disparity is statistically reliable, not random noise.
- **Effect size (Risk Ratio)** — ratio of the minority group's approval rate to the majority group's. Values below 0.8 are typically flagged as discriminatory under the 80% rule.

---

## 4. Step 2 — Data transformation and fair training

Two sub-steps run sequentially:

1. **`DisparateImpactRemover`** (PipelineModule) repairs numeric feature distributions across groups, reducing proxy discrimination in the input data.
2. **`ReductionsWrapper`** (TrainingModule) trains a `LogisticRegression` using fairlearn's `ExponentiatedGradient` algorithm, enforcing a `DemographicParity` constraint throughout optimisation.

In [8]:
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from run_pipeline import step2_transform_and_train

model, X_test_tr, y_test, s_test = step2_transform_and_train(df, cfg)


STEP 2 — DATA TRANSFORMATION & FAIR TRAINING

  Applying DisparateImpactRemover (repair_level=0.8)...
  ✓ Transformation complete — feature shape: (41600, 16)

  Training ReductionsWrapper with DemographicParity constraint...

=== ReductionsWrapper Summary ===
Constraint    : DemographicParity
eps           : 0.01
Predictors    : 3
Top weights   : [(np.float64(1.0), 2), (np.float64(0.0), 1), (np.float64(0.0), 0)]
Best estimator: predictor[2] (weight=1.0000)


In [9]:
import numpy as np

print("Ensemble summary:")
print(f"  Number of predictors : {len(model.predictors_)}")
print(f"  Best predictor weight: {model.best_weight_:.4f}")
print(f"  Top-3 weights        : {sorted(model.weights_, reverse=True)[:3]}")

Ensemble summary:
  Number of predictors : 3
  Best predictor weight: 1.0000
  Top-3 weights        : [np.float64(1.0), np.float64(0.0), np.float64(0.0)]


---

## 5. Step 3 — Final validation and report card

`FairnessAnalyzer` re-evaluates on the **test-set predictions** of the fair model.  
The report card compares the final metric against the baseline and the configured threshold.

In [10]:
from run_pipeline import step3_validate

final_results, accuracy = step3_validate(
    model, X_test_tr, y_test, s_test, cfg, baseline_results
)


STEP 3 — FINAL VALIDATION & REPORT CARD

  Accuracy on test set: 0.8270
FairnessAnalyzer ready — 10,400 rows loaded.

  ┌──────────────────────────────────────────────────────────┐
  │                   FAIRNESS REPORT CARD                   │
  └──────────────────────────────────────────────────────────┘

  Metric    : demographic_parity_difference
  Baseline  : 0.0040  CI (0.0003, 0.0120)
  Final     : 0.0020  CI (0.0003, 0.0201)
  Change    : -0.0020
  Threshold : ≤ 0.1

  Status    : ✅ PASS


In [11]:
# Side-by-side summary table
f = final_results[primary]
b = baseline_results[primary]

comparison = pd.DataFrame({
    "Stage":      ["Baseline (raw labels)", "Final (model predictions)"],
    "Value":      [round(b.value, 4),  round(f.value, 4)],
    "CI lower":   [round(b.confidence_interval[0], 4), round(f.confidence_interval[0], 4)],
    "CI upper":   [round(b.confidence_interval[1], 4), round(f.confidence_interval[1], 4)],
    "Effect size":[round(b.effect_size, 4) if b.effect_size else None,
                   round(f.effect_size, 4) if f.effect_size else None],
})

threshold = cfg["validation"]["threshold"]
delta     = f.value - b.value
passed    = abs(f.value) <= threshold

print(f"Primary metric   : {primary}")
print(f"Change           : {delta:+.4f}")
print(f"Threshold        : ≤ {threshold}")
print(f"Validation gate  : {'✅ PASS' if passed else '❌ FAIL'}")
print(f"Test accuracy    : {accuracy:.4f}")
print()
comparison

Primary metric   : demographic_parity_difference
Change           : -0.0020
Threshold        : ≤ 0.1
Validation gate  : ✅ PASS
Test accuracy    : 0.8270



,Stage,Value,CI lower,CI upper,Effect size
0,Baseline (raw labels),0.004,0.0003,0.0120,0.9938
1,Final (model predictions),0.002,0.0003,0.0201,0.9973


---

## 6. MLflow — log everything

A single call logs:
- **Metrics** — accuracy, baseline and final fairness metric with CI bounds
- **Params** — transformer, constraint, threshold
- **Artifacts** — the trained model and the exact `config.yml` used

In [12]:
from run_pipeline import log_to_mlflow

log_to_mlflow(model, baseline_results, final_results, accuracy, cfg, CONFIG_PATH)


LOGGING TO MLFLOW


2026/05/26 16:14:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/26 16:14:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  ✓ Run logged to experiment 'fairness_pipeline_run'.
  Run  mlflow ui  to explore metrics and artifacts.


In [13]:
# Preview what was logged
import mlflow

client = mlflow.MlflowClient()
exp    = client.get_experiment_by_name(cfg["mlflow"]["experiment_name"])
runs   = client.search_runs(experiment_ids=[exp.experiment_id],
                             order_by=["start_time DESC"],
                             max_results=1)

if runs:
    run = runs[0]
    print(f"Run ID   : {run.info.run_id}")
    print(f"Run name : {run.data.tags.get('mlflow.runName')}")
    print(f"Status   : {run.info.status}")
    print("\nLogged metrics:")
    for k, v in sorted(run.data.metrics.items()):
        print(f"  {k:<45} {v:.6f}")
    print("\nLogged params:")
    for k, v in sorted(run.data.params.items()):
        print(f"  {k:<30} {v}")

Run ID   : 0914c2325131448c928859f538bc1b32
Run name : gender_demographic_parity
Status   : FINISHED

Logged metrics:
  accuracy                                      0.827019
  baseline_demographic_parity_difference        0.004004
  final_demographic_parity_difference           0.001996
  final_demographic_parity_difference_ci_lower  0.000271
  final_demographic_parity_difference_ci_upper  0.020098
  final_demographic_parity_difference_effect_size 0.997269

Logged params:
  constraint                     DemographicParity
  fairness_threshold             0.1
  sensitive_col                  Gender_Male
  training_method                ReductionsWrapper
  transformer                    DisparateImpactRemover


Open the MLflow UI to explore the run visually:

```bash
mlflow ui
# → http://localhost:5000
```

Navigate to the experiment, open the run, and check the **Artifacts** tab to see the logged model and config file.

---

## 7. Run the full pipeline in one call

All of the above is orchestrated by a single entry point.  
In production, a team member runs this — no notebook needed.

In [14]:
from run_pipeline import main

main(config_path=CONFIG_PATH)

✓ Data loaded: 52,000 rows × 27 columns.

STEP 1 — BASELINE FAIRNESS AUDIT
FairnessAnalyzer ready — 52,000 rows loaded.

  Demographic Parity Difference
  Value       : 0.0040
  95% CI      : (0.0003, 0.0120)
  Effect size : 0.9937785536822049
  Groups (n)  : {1: 26011, 0: 25989}

STEP 2 — DATA TRANSFORMATION & FAIR TRAINING

  Applying DisparateImpactRemover (repair_level=0.8)...
  ✓ Transformation complete — feature shape: (41600, 16)

  Training ReductionsWrapper with DemographicParity constraint...

=== ReductionsWrapper Summary ===
Constraint    : DemographicParity
eps           : 0.01
Predictors    : 3
Top weights   : [(np.float64(1.0), 2), (np.float64(0.0), 1), (np.float64(0.0), 0)]
Best estimator: predictor[2] (weight=1.0000)

STEP 3 — FINAL VALIDATION & REPORT CARD

  Accuracy on test set: 0.8270
FairnessAnalyzer ready — 10,400 rows loaded.


2026/05/26 16:18:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



  ┌──────────────────────────────────────────────────────────┐
  │                   FAIRNESS REPORT CARD                   │
  └──────────────────────────────────────────────────────────┘

  Metric    : demographic_parity_difference
  Baseline  : 0.0040  CI (0.0003, 0.0120)
  Final     : 0.0020  CI (0.0003, 0.0201)
  Change    : -0.0020
  Threshold : ≤ 0.1

  Status    : ✅ PASS

LOGGING TO MLFLOW


2026/05/26 16:18:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  ✓ Run logged to experiment 'fairness_pipeline_run'.
  Run  mlflow ui  to explore metrics and artifacts.

✓ Pipeline complete.



Or from the terminal:

```bash
python run_pipeline.py --config config.yml
```

---

## 8. Key takeaways

| | |
|---|---|
| **Declarative config** | The entire workflow is reproducible from `config.yml` alone — no hardcoded values anywhere in the orchestrator |
| **Two-layer mitigation** | Pre-processing (`DisparateImpactRemover`) and in-processing (`ReductionsWrapper`) work together; neither alone is sufficient |
| **Statistical rigour** | Every fairness metric comes with a bootstrapped 95% CI, so the report card distinguishes real improvement from sampling noise |
| **MLflow traceability** | Every run is fully logged — metrics, params, model artifact, and the exact config used — making results comparable across the organisation |
| **Extendable** | Adding a new transformer or constraint is one line in the registry dict and one line in `config.yml` |